# Dark Factory MAS — разбор данных и демонстрация цикла

Ноутбук воспроизводит ключевые выводы проекта и показывает один полный цикл
принятия решения. Он **не содержит логики** — вся логика в `src/dfmas/`,
ноутбук только вызывает её. Так исключается расхождение между «что показали»
и «что работает».

Перед запуском: `make data && make fit`.


In [1]:
import sys, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 40)


## 1. Данные: что реально лежит в выгрузках


In [2]:
from dfmas.io.ingest import load_processed
tele, flags, lab = load_processed()
print('телеметрия:', tele.shape, tele.index.min(), '—', tele.index.max())
print('анализы:', lab.groupby('source').size().to_dict())
lab.groupby(['source','stream','param']).size().head(20)


телеметрия: (189217, 97) 2023-01-01 00:00:00 — 2026-08-07 00:00:00
анализы: {'LIMS': 36800, 'PAK': 189649}


source  stream          param       
LIMS    AVT_DIESEL_P1   50%.T            648
                        90%.T            580
                        95%.T            649
                        CFPP              44
                        CloudPoint        13
                        CloudPoint_1     112
                        D15              219
                        EBP.T            648
                        I350             138
                        IBP.T            648
                        PourPoint       1255
        AVT_DIESEL_P2   50%.T            212
                        90%.T            206
                        95%.T            188
                        CloudPoint        20
                        D15              120
                        EBP.T            212
                        IBP.T            212
        AVT_DIESEL_P21  50%.T            191
                        90%.T            183
dtype: int64

### Сентинел 307 и прочие коды историка

Историк не помечает плохие значения пропуском — он подставляет числовые коды.
Эксперт подтвердил, что 307 — выброс. Мы дополнительно ищем «цифровые состояния»
прибора автоматически, без списка магических чисел в коде.


In [3]:
raw = pd.read_csv('../data/raw/242000_tags.csv', index_col=0, parse_dates=['date']).set_index('date')
share = (raw == 307.0).mean().sort_values(ascending=False) * 100
print('доля сентинела 307 по тегам, %:')
print(share.head(8).round(2).to_string())

from dfmas.quality.sentinels import FLAG_NAMES
cnt = pd.Series(flags.to_numpy().ravel()).value_counts(normalize=True) * 100
print()
print({FLAG_NAMES[int(k)]: round(v, 2) for k, v in cnt.items()})


доля сентинела 307 по тегам, %:
Q20    11.65
Q21     2.97
F2      2.88
T16     1.52
F17     1.23
F19     1.23
F9      1.22
T12     0.91

{'good': 96.62, 'sentinel': 1.99, 'flatline': 0.49, 'out_of_range': 0.48, 'digital_state': 0.42}


### Какой тег на самом деле показывает серу

Справочник приписывает серу тегам T6, W7 и P13. Проверяем по лаборатории.


In [4]:
lp = lab[(lab.stream=='HT_PRODUCT') & (lab.param=='Mg.Sulfur') & (lab.source=='LIMS')]
lp = lp[(lp.value>0.2)&(lp.value<60)].drop_duplicates(subset='measured_at')
y = pd.Series(lp.value.to_numpy(), index=pd.DatetimeIndex(lp.measured_at)).sort_index()
rows=[]
for tag in ['H_Q21','H_W7','H_P13','H_T6']:
    s = tele[tag].rolling(6, min_periods=2).median().dropna()
    pred = pd.Series(np.interp(y.index.view('i8'), s.index.view('i8'), s.to_numpy()), index=y.index)
    d = pd.DataFrame({'tag':pred,'lab':y}).dropna()
    rows.append({'тег':tag,'корреляция':round(d.tag.corr(d.lab),3),
                 'MAE':round((d.tag-d.lab).abs().mean(),2)})
pd.DataFrame(rows)


,тег,корреляция,MAE
0,H_Q21,0.412,1.45
1,H_W7,-0.047,8.36
2,H_P13,-0.062,4.71
3,H_T6,0.061,355.38


Ноль корреляции у T6, W7, P13 и заметная связь у Q21 — подтверждение сдвига
строк в листе «КИП». Полный разбор: `reports/01_tag_audit.md`.


## 2. Витрина «на момент времени»

Единственный способ получить данные в системе. Обратите внимание на **возраст**
каждого анализа — именно он определяет, можно ли на него опираться.


In [5]:
from dfmas.featurestore import AsOfStore
store = AsOfStore.load()
state = store.snapshot('2025-11-20 14:00:00')
print('сера продукта (поточный):', round(state.mean('H_Q21', 60), 2), 'мг/кг')
print()
for k, v in sorted(state.lab.items()):
    if k[0] in ('HT_FEED','HT_PRODUCT') and k[1] in ('Mass.Sulfur','Mg.Sulfur','95%.T','CetaneNumber'):
        print(f'{k[0]:11s} {k[1]:13s} {v.value:9.3f} {v.unit:7s} {v.source:5s} возраст {v.age_hours:8.1f} ч')


сера продукта (поточный): 8.96 мг/кг

HT_FEED     95%.T           352.000 degC    LIMS  возраст      4.0 ч
HT_FEED     Mass.Sulfur       0.935 wt%     LIMS  возраст    124.0 ч
HT_PRODUCT  95%.T           350.000 degC    LIMS  возраст      4.0 ч
HT_PRODUCT  CetaneNumber     54.600 cetane  LIMS  возраст    364.0 ч
HT_PRODUCT  Mg.Sulfur         6.100 mg/kg   LIMS  возраст      4.0 ч
HT_PRODUCT  Mg.Sulfur         6.125 mg/kg   PAK   возраст      0.2 ч


## 3. Виртуальные анализаторы: формулы эксперта + коррекция смещения


In [6]:
from dfmas.models import va_formulas as va
res = va.evaluate(state)
pd.DataFrame([{'ВАК':k,'значение':round(v.value,2) if np.isfinite(v.value) else None,
               'применим':v.valid,'чего не хватает':'; '.join(v.missing[:2])}
              for k,v in res.items()])


,ВАК,значение,применим,чего не хватает
0,24-2000:GODT:T90,346.87,True,
1,24-2000:GODT:T50,275.34,True,
2,24-2000:GODT:I250,18.15,True,
3,24-2000:GODT:D15,NaN,False,LIMS:HT_FEED:D15(устарел 3076 ч)
4,24-2000:GODT:CloudPoint,1.10,True,
5,24-2000:GODT:T95,352.68,True,
6,24-2000:GODT:CFPP,-16.91,True,
7,24-2000:GODT:IBP,193.97,True,
8,AVT6:240-350:D15,865.54,True,
9,AVT6:240-350:T50,279.04,True,


## 4. Кинетика гидрообессеривания — физический приор

Монотонна по построению, решает обратную задачу «какая температура нужна».


In [7]:
from dfmas.models.hds import HDSParams, calibrate_k0, required_temperature, sulfur_out
from dfmas.models.fit import ModelBundle
bundle = ModelBundle.load(); hds = HDSParams(**bundle.hds); ref = bundle.reference
print('опорная точка:', {k: round(v,3) for k,v in ref.items() if isinstance(v,float)})
pd.DataFrame([{'сера сырья, % масс.': s,
               'сера продукта при текущем T': round(sulfur_out(s*1e4, ref['t_reactor_c'], ref['lhsv'], ref['p_mpa'], hds), 2),
               'T для цели 8 мг/кг': round(required_temperature(8.0, s*1e4, ref['lhsv'], ref['p_mpa'], hds), 2)}
              for s in (0.80, 0.95, 1.10, 1.25, 1.40)])


опорная точка: {'s_feed_wt': 0.951, 't_reactor_c': 368.99, 'p_mpa': 4.317, 'load': 259.305, 'h2': 0.585, 'lhsv': 1.5, 's_out': 8.578, 'p8_units_per_degC': 0.001, 'temp_p5': 358.446, 'temp_p95': 384.881, 'load_p5': 219.638, 'load_p95': 293.246, 'press_p5': 3.655, 'press_p95': 4.888, 'dp_p95': 4.656, 'dp_median': 2.299, 'quench_median': 3397.146, 'scale_temp_set': 2000.0, 'scale_press': 0.02}


,"сера сырья, % масс.",сера продукта при текущем T,T для цели 8 мг/кг
0,0.80,7.30,367.44
1,0.95,8.57,370.14
2,1.10,9.93,372.65
3,1.25,11.40,374.99
4,1.40,12.97,377.20


## 5. Полный цикл мультиагентной системы


In [8]:
from dfmas.system import DarkFactorySystem
system = DarkFactorySystem.build()
result = system.run_at('2025-11-20 14:00:00', grade='DT_SUMMER')
print(result.recommendation.expected_effect['_markdown'])


### Время и состояние
Срез **2025-11-20 14:00**. Пригодность данных **0.80** — данных достаточно.
Свежесть источников — LIMS_feed_sulfur: 124.0 ч, LIMS_product_sulfur: 4.0 ч, PAK: 0.2 ч, TELEMETRY: 0.0 ч.
Сырьё гидроочистки: сера **0.935 % масс.** (ЛИМС (устаревший), возраст 124 ч), Т95 352.0 °C, расход 96.5 т/ч.

### Проблема / риск
Сера продукта сейчас **6.10 мг/кг** (ЛИМС (контрольный факт)); прогноз через 120 мин **без вмешательства** — **6.10** мг/кг, интервал 90 % [2.43; 9.20].
Вероятность выхода за предел 10 мг/кг при бездействии — **3 %**.
Чтобы риск опустился до 10 %, уровень серы должен быть не выше **7.71 мг/кг** (предел 10 минус 90-й процентиль разброса измерения).
- Данные: анализ серы сырья устарел: 124 ч при допустимых 36 ч.

### Предлагаемое действие
**температура реактора -2.7 °C, загрузка -3.0 %, давление -0.20 МПа, расход ВСГ -2.7 %**
- температура реактора (H_T5 — ГСС на выходе Р-201): -2.7 °C; эквивалент по уставке H_P8 (Р-202, температура ГСС на входе): -0.0018 ед

### Журнал обмена агентов — «логика принятия решения»


In [9]:
for m in result.trace:
    print(f"{m['seq']:3d}  {m['sender']:13s} -> {m['recipient']:13s}  {m['topic']:26s} {m['elapsed_ms']:7.1f} мс")


  1  orchestrator  -> data           assess.request                 0.0 мс
  2  data          -> orchestrator   assess.response                2.8 мс
  3  orchestrator  -> feed           assess.request                 0.0 мс
  4  feed          -> orchestrator   assess.response                1.9 мс
  5  orchestrator  -> quality        assess.request                 0.0 мс
  6  quality       -> orchestrator   assess.response                1.6 мс
  7  orchestrator  -> reliability    assess.request                 0.0 мс
  8  reliability   -> orchestrator   assess.response                0.1 мс
  9  orchestrator  -> optimizer      optimize.request               0.0 мс
 10  optimizer     -> *              optimizer.summary              0.0 мс
 11  optimizer     -> orchestrator   optimize.response            139.4 мс
 12  orchestrator  -> *              conflict.resolution            0.0 мс
 13  orchestrator  -> *              setpoint.plan                  0.0 мс
 14  orchestrator  -> ble

## 6. Сценарии не «жёсткие»: меняем серу сырья и смотрим на решение


In [10]:
from dfmas.sweep import feed_sulfur_sweep
sw = feed_sulfur_sweep(system, '2024-04-26 20:00:00')
sw[['сера сырья, % масс.','решение','ΔT, °C','прогноз при бездействии, мг/кг','риск, %','требуемая компенсация, °C']]


,"сера сырья, % масс.",решение,"ΔT, °C","прогноз при бездействии, мг/кг","риск, %","требуемая компенсация, °C"
0,0.70,act,-2.6510,5.698827,3.3,-7.891070
1,0.85,act,-2.6510,6.172375,4.2,-5.035702
2,1.00,act,-2.6510,6.643220,6.7,-2.396955
3,1.15,act,-2.6510,7.111868,9.8,0.059703
4,1.30,act,-2.6510,7.578632,9.4,2.360288
5,1.45,hold_best_effort,0.0000,8.043729,12.1,4.525166
6,1.60,act_best_effort,0.8837,8.507315,12.1,6.570711
7,1.80,act_best_effort,2.6510,9.123283,13.2,9.135353


Прогноз растёт монотонно, требуемая компенсация меняет знак: от «снизить
температуру и сэкономить» до «поднять, иначе нарушим норматив». Решение
системы меняется вслед за входными данными — ровно то, чего требовал эксперт.


## 7. Воспроизводимость


In [11]:
a = system.run_at('2025-11-20 14:00:00').recommendation
b = system.run_at('2025-11-20 14:00:00').recommendation
print('отпечаток состояния совпадает:', a.state_hash == b.state_hash)
print('текст рекомендации совпадает :', a.headline == b.headline)
print('числа совпадают              :', json.dumps(a.as_dict(), sort_keys=True, default=str) ==
      json.dumps(b.as_dict(), sort_keys=True, default=str))


отпечаток состояния совпадает: True
текст рекомендации совпадает : True
числа совпадают              : True
